# 01 — EDA Preprocessed & Feature Engineering

Builds a leakage-safe feature store. Future rows use calendar, raw-derived promotion projections, seasonal profiles, and shifted target history profiles; they do not use same-day future transactions.


In [ ]:
from pathlib import Path
import json
import random
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
RANDOM_SEED = 2026
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

def find_package_paths():
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        # Final GitHub layout: repo_root/Raw Data and repo_root/part3_forecasting/notebooks
        if (candidate / "Raw Data").exists() and (candidate / "part3_forecasting" / "notebooks").exists():
            return candidate, candidate / "part3_forecasting"
        # Backward-compatible layout used during development: package_root/Raw Data and package_root/notebooks
        if (candidate / "Raw Data").exists() and (candidate / "notebooks").exists():
            return candidate, candidate
    raise FileNotFoundError("Cannot locate repository root with Raw Data and forecasting notebooks")

REPO_ROOT, PACKAGE_ROOT = find_package_paths()
RAW_DIR = REPO_ROOT / "Raw Data"
NB_DIR = PACKAGE_ROOT / "notebooks"
ARTIFACT_DIR = PACKAGE_ROOT / "artifacts"
REPORT_DIR = PACKAGE_ROOT / "reports"
OUTPUT_DIR = PACKAGE_ROOT / "outputs"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("PACKAGE_ROOT =", PACKAGE_ROOT)
print("RAW_DIR =", RAW_DIR)


## Load Raw Data

In [ ]:
def read_csv(name, parse_dates=None, usecols=None, dtype=None):
    return pd.read_csv(RAW_DIR / name, parse_dates=parse_dates or [], usecols=usecols, dtype=dtype, low_memory=False)

sales = read_csv("sales.csv", parse_dates=["Date"]).sort_values("Date")
sample = read_csv("sample_submission.csv", parse_dates=["Date"], usecols=["Date"])
orders = read_csv("orders.csv", parse_dates=["order_date"])
order_items = read_csv("order_items.csv", dtype={"promo_id": "string", "promo_id_2": "string"})
products = read_csv("products.csv")
payments = read_csv("payments.csv")
promotions = read_csv("promotions.csv", parse_dates=["start_date", "end_date"])
returns = read_csv("returns.csv", parse_dates=["return_date"])
inventory = read_csv("inventory.csv", parse_dates=["snapshot_date"])
web = read_csv("web_traffic.csv", parse_dates=["date"])

assert sample.columns.tolist() == ["Date"]
assert len(sample) == 548
print(sales.shape, sales["Date"].min(), sales["Date"].max())


## Calendar, Tet, Black Friday, Double-Day, Fashion Features

In [ ]:
TET_DATES = {
    2013: "2013-02-10", 2014: "2014-01-31", 2015: "2015-02-19", 2016: "2016-02-08",
    2017: "2017-01-28", 2018: "2018-02-16", 2019: "2019-02-05", 2020: "2020-01-25",
    2021: "2021-02-12", 2022: "2022-02-01", 2023: "2023-01-22", 2024: "2024-02-10",
}

def fourth_friday_nov(year):
    days = pd.date_range(f"{year}-11-01", f"{year}-11-30", freq="D")
    fridays = days[days.dayofweek == 4]
    return fridays[3]


def add_vietnamese_calendar_features(df):
    """Add compact Vietnam retail calendar features using vectorized date operations.

    `days_to_tet` uses the same sign convention as `days_from_tet`: negative before
    Tet, positive after Tet. Known dates come from `TET_DATES`; years outside the
    mapping are approximated with a 354.37-day lunar-year step so the function
    remains well-defined outside the current train/test horizon.
    """
    out = df.copy()
    dates = pd.to_datetime(out["Date"])
    min_year = int(dates.dt.year.min())
    max_year = int(dates.dt.year.max())

    tet_by_year = {int(y): pd.Timestamp(d) for y, d in TET_DATES.items()}
    known_years = sorted(tet_by_year)
    lunar_year_days = 354.37

    for year in range(min_year - 1, max_year + 2):
        if year in tet_by_year:
            continue
        if year < known_years[0]:
            offset = int(round((known_years[0] - year) * lunar_year_days))
            tet_by_year[year] = tet_by_year[known_years[0]] - pd.Timedelta(days=offset)
        elif year > known_years[-1]:
            offset = int(round((year - known_years[-1]) * lunar_year_days))
            tet_by_year[year] = tet_by_year[known_years[-1]] + pd.Timedelta(days=offset)

    tet_candidates = np.array([
        ts.to_datetime64().astype("datetime64[D]").astype(int)
        for _, ts in sorted(tet_by_year.items())
    ])
    date_int = dates.values.astype("datetime64[D]").astype(int)
    diffs = date_int[:, None] - tet_candidates[None, :]
    out["days_to_tet"] = diffs[np.arange(len(out)), np.argmin(np.abs(diffs), axis=1)].astype(int)

    # Non-overlapping pre-Tet windows: -7 belongs to tet_pre_14, matching the audit test.
    out["tet_pre_14"] = out["days_to_tet"].between(-14, -7).astype(int)
    out["tet_pre_7"] = out["days_to_tet"].between(-6, -1).astype(int)
    out["tet_post_7"] = out["days_to_tet"].between(1, 7).astype(int)

    month = dates.dt.month
    day = dates.dt.day
    out["is_vn_major_holiday"] = (
        ((month == 4) & (day == 30)) |
        ((month == 5) & (day == 1)) |
        ((month == 9) & (day == 2))
    ).astype(int)
    out["vn_payday_cycle"] = day.isin([1, 2, 3, 24, 25, 26, 27]).astype(int)

    out["fashion_micro_season"] = np.select(
        [
            month.isin([1, 2]),
            month.isin([3, 4]),
            month.isin([5, 6]),
            month.isin([7, 8]),
            month.isin([9, 10]),
            month.isin([11, 12]),
        ],
        [1, 2, 3, 4, 5, 6],
        default=0,
    ).astype(int)
    return out


def add_sale_event_features(df):
    """Add Vietnam e-commerce mega-sale event flags and countdown windows.

    Events are public commerce-calendar dates. The implementation uses vectorized
    date arithmetic and annual event candidates for the train/test years.
    """
    out = df.copy()
    dates = pd.to_datetime(out["Date"])
    month = dates.dt.month
    day = dates.dt.day

    out["is_1111"] = ((month == 11) & (day == 11)).astype(int)
    out["is_1212"] = ((month == 12) & (day == 12)).astype(int)
    out["is_828"] = ((month == 8) & (day == 28)).astype(int)

    date_int = dates.values.astype("datetime64[D]").astype(int)
    years = np.arange(int(dates.dt.year.min()) - 1, int(dates.dt.year.max()) + 2)
    for event_name, event_month, event_day in [("1111", 11, 11), ("1212", 12, 12)]:
        event_candidates = np.array([
            pd.Timestamp(year=int(year), month=event_month, day=event_day)
            .to_datetime64()
            .astype("datetime64[D]")
            .astype(int)
            for year in years
        ])
        diffs = date_int[:, None] - event_candidates[None, :]
        days_to_event = diffs[np.arange(len(out)), np.argmin(np.abs(diffs), axis=1)].astype(int)
        out[f"days_to_{event_name}"] = days_to_event
        out[f"{event_name}_pre_7"] = pd.Series(days_to_event).between(-7, -1).astype(int).to_numpy()
        out[f"{event_name}_pre_3"] = pd.Series(days_to_event).between(-3, -1).astype(int).to_numpy()
        out[f"{event_name}_day"] = (days_to_event == 0).astype(int)
        out[f"{event_name}_post_3"] = pd.Series(days_to_event).between(1, 3).astype(int).to_numpy()
    return out

def add_calendar(frame):
    frame = frame.copy()
    d = frame["Date"]
    frame["year"] = d.dt.year
    frame["month"] = d.dt.month
    frame["day"] = d.dt.day
    frame["day_of_week"] = d.dt.dayofweek
    frame["day_of_year"] = d.dt.dayofyear
    frame["week_of_year"] = d.dt.isocalendar().week.astype(int)
    frame["quarter"] = d.dt.quarter
    frame["is_weekend"] = (frame["day_of_week"] >= 5).astype(int)
    frame["is_month_start"] = d.dt.is_month_start.astype(int)
    frame["is_month_end"] = d.dt.is_month_end.astype(int)
    frame["days_in_month"] = d.dt.days_in_month
    frame["week_of_month"] = ((frame["day"] - 1) // 7 + 1).astype(int)
    frame["day_position_in_month"] = frame["day"] / frame["days_in_month"]
    for period, col in [(7, "dow"), (12, "month"), (365.25, "doy")]:
        value = {"dow": frame["day_of_week"], "month": frame["month"], "doy": frame["day_of_year"]}[col]
        frame[f"sin_{col}"] = np.sin(2 * np.pi * value / period)
        frame[f"cos_{col}"] = np.cos(2 * np.pi * value / period)
    for k in range(2, 6):
        frame[f"sin_doy_k{k}"] = np.sin(2 * np.pi * k * frame["day_of_year"] / 365.25)
        frame[f"cos_doy_k{k}"] = np.cos(2 * np.pi * k * frame["day_of_year"] / 365.25)
    frame["is_first_3_days"] = (frame["day"] <= 3).astype(int)
    frame["is_last_3_days"] = (frame["day"] >= frame["days_in_month"] - 2).astype(int)
    frame["days_from_nearest_payday"] = np.minimum((frame["day"] - 5).abs(), (frame["day"] - 25).abs()).clip(0, 10)
    frame["is_private_payday"] = frame["day"].between(5, 10).astype(int)
    frame["is_government_payday"] = frame["day"].between(24, 26).astype(int)
    frame["is_spring_collection"] = frame["month"].isin([2, 3, 4]).astype(int)
    frame["is_summer_sale"] = frame["month"].isin([6, 7]).astype(int)
    frame["is_fall_collection"] = frame["month"].isin([8, 9]).astype(int)
    frame["is_winter_clearance"] = frame["month"].isin([12, 1]).astype(int)
    frame["is_back_to_school"] = frame["month"].isin([8, 9]).astype(int)
    frame["is_yearend_shopping"] = frame["month"].isin([11, 12]).astype(int)
    frame["month_x_weekday"] = frame["month"] * 10 + frame["day_of_week"]
    frame["quarter_x_weekend"] = frame["quarter"] * frame["is_weekend"]
    frame = add_vietnamese_calendar_features(frame)
    frame = add_sale_event_features(frame)

    tet_candidates = np.array([pd.Timestamp(v).to_datetime64().astype("datetime64[D]").astype(int) for v in TET_DATES.values()])
    date_int = d.values.astype("datetime64[D]").astype(int)
    diffs = date_int[:, None] - tet_candidates[None, :]
    nearest = diffs[np.arange(len(frame)), np.argmin(np.abs(diffs), axis=1)]
    frame["days_from_tet"] = nearest.clip(-120, 120)
    frame["tet_proximity"] = np.exp(-np.abs(frame["days_from_tet"]) / 18)
    windows = {
        "tet_pre_28_15": (-28, -15), "tet_pre_14_8": (-14, -8), "tet_pre_7_1": (-7, -1),
        "tet_0_6": (0, 6), "tet_post_7_14": (7, 14), "tet_post_15_28": (15, 28),
        "tet_post_29_56": (29, 56), "tet_post_57_110": (57, 110),
    }
    for name, (lo, hi) in windows.items():
        frame[name] = frame["days_from_tet"].between(lo, hi).astype(int)

    dd_candidates = []
    for y in sorted(frame["year"].unique()):
        for m in range(1, 13):
            dd_candidates.append(pd.Timestamp(year=int(y), month=m, day=m))
    dd_int = np.array([x.to_datetime64().astype("datetime64[D]").astype(int) for x in dd_candidates])
    dd_diffs = date_int[:, None] - dd_int[None, :]
    nearest_dd = dd_diffs[np.arange(len(frame)), np.argmin(np.abs(dd_diffs), axis=1)]
    frame["days_from_nearest_double_day"] = nearest_dd.clip(-30, 30)
    frame["is_date_eq_month"] = (frame["month"] == frame["day"]).astype(int)
    frame["is_double_day_window_3"] = (frame["days_from_nearest_double_day"].abs() <= 3).astype(int)
    frame["is_modern_commerce_double_day"] = ((frame["month"].isin([9, 10, 11, 12])) & (frame["month"] == frame["day"])).astype(int)

    bf_candidates = np.array([fourth_friday_nov(y).to_datetime64().astype("datetime64[D]").astype(int) for y in sorted(frame["year"].unique())])
    bf_diffs = date_int[:, None] - bf_candidates[None, :]
    nearest_bf = bf_diffs[np.arange(len(frame)), np.argmin(np.abs(bf_diffs), axis=1)]
    frame["days_from_black_friday"] = nearest_bf.clip(-45, 45)
    frame["black_friday_proximity"] = np.exp(-np.abs(frame["days_from_black_friday"]) / 10)
    frame["is_black_friday_actual"] = (frame["days_from_black_friday"] == 0).astype(int)
    frame["is_black_friday_pre_7_1"] = frame["days_from_black_friday"].between(-7, -1).astype(int)
    frame["is_black_friday_post_1_7"] = frame["days_from_black_friday"].between(1, 7).astype(int)
    return frame

all_dates = pd.DataFrame({"Date": pd.date_range("2013-01-01", sample["Date"].max(), freq="D")})
features = all_dates.merge(sales[["Date", "Revenue", "COGS"]], on="Date", how="left")
features["COGS_ratio"] = features["COGS"] / features["Revenue"].replace(0, np.nan)
features = add_calendar(features)
features["month_day"] = features["Date"].dt.strftime("%m-%d")
features.head()


In [ ]:
# Unit test for compact Vietnamese calendar features.
_vn_calendar_test = add_vietnamese_calendar_features(pd.DataFrame({"Date": [pd.Timestamp("2023-01-15")]}))
assert int(_vn_calendar_test.loc[0, "days_to_tet"]) == -7
assert int(_vn_calendar_test.loc[0, "tet_pre_14"]) == 1
assert int(_vn_calendar_test.loc[0, "tet_pre_7"]) == 0
display(_vn_calendar_test)


In [ ]:
# Unit test for mega sale event features.
_sale_event_test = add_sale_event_features(pd.DataFrame({"Date": pd.to_datetime(["2022-11-08", "2022-11-11", "2022-11-14"])}))
assert int(_sale_event_test.loc[0, "1111_pre_3"]) == 1
assert int(_sale_event_test.loc[1, "is_1111"]) == 1
assert int(_sale_event_test.loc[1, "1111_day"]) == 1
assert int(_sale_event_test.loc[2, "1111_post_3"]) == 1
display(_sale_event_test[["Date", "is_1111", "1111_pre_3", "1111_day", "1111_post_3"]])


## Raw-Derived Promotion Features

In [ ]:
def normalize_campaign(name):
    txt = str(name).lower()
    for y in range(2012, 2025):
        txt = txt.replace(str(y), "")
    return " ".join(txt.replace("-", " ").split())

def campaign_bucket(name):
    txt = normalize_campaign(name)
    if "spring" in txt: return "spring"
    if "mid" in txt or "year" in txt and "end" not in txt: return "mid_year"
    if "fall" in txt: return "fall"
    if "end" in txt or "year end" in txt: return "year_end"
    if "urban" in txt: return "urban"
    if "rural" in txt: return "rural"
    return "other"

def project_promotions(raw_promos, years=(2023, 2024)):
    p = raw_promos.copy()
    p["campaign_key"] = p["promo_name"].map(normalize_campaign)
    p["duration"] = (p["end_date"] - p["start_date"]).dt.days.clip(lower=0)
    rows = []
    for _, g in p.sort_values("start_date").groupby("campaign_key"):
        if len(g) < 2:
            continue
        template = g.iloc[-1]
        start_month, start_day = int(template["start_date"].month), int(template["start_date"].day)
        for year in years:
            start = pd.Timestamp(year=year, month=start_month, day=min(start_day, 28 if start_month == 2 else start_day))
            rows.append({
                "promo_id": f"projected_{template['campaign_key']}_{year}",
                "promo_name": f"{template['campaign_key']} projected {year}",
                "promo_type": template["promo_type"],
                "discount_value": template["discount_value"],
                "start_date": start,
                "end_date": start + pd.Timedelta(days=int(template["duration"])),
                "promo_channel": template.get("promo_channel", ""),
                "stackable_flag": template.get("stackable_flag", 0),
                "applicable_category": template.get("applicable_category", None),
            })
    return pd.concat([raw_promos, pd.DataFrame(rows)], ignore_index=True, sort=False)

def build_promo_features(dates, raw_promos):
    promos = project_promotions(raw_promos)
    promos["campaign_bucket"] = promos["promo_name"].map(campaign_bucket)
    long_rows = []
    for _, r in promos.iterrows():
        for dt in pd.date_range(r["start_date"], r["end_date"], freq="D"):
            long_rows.append({
                "Date": dt, "promo_count": 1,
                "promo_discount_mean": float(r.get("discount_value", 0) or 0),
                "promo_discount_max": float(r.get("discount_value", 0) or 0),
                "promo_stackable_count": int(r.get("stackable_flag", 0) or 0),
                "promo_percentage_count": int(str(r.get("promo_type", "")).lower().startswith("percent")),
                "promo_fixed_count": int(str(r.get("promo_type", "")).lower().startswith("fixed")),
                "promo_online_count": int("online" in str(r.get("promo_channel", "")).lower()),
                "promo_all_channel_count": int("all" in str(r.get("promo_channel", "")).lower()),
                f"promo_{r['campaign_bucket']}_active": 1,
                "promo_start": r["start_date"],
                "promo_end": r["end_date"],
            })
    if not long_rows:
        return dates[["Date"]].copy()
    long = pd.DataFrame(long_rows)
    bucket_cols = [c for c in long.columns if c.startswith("promo_") and c.endswith("_active")]
    agg = {c: "sum" for c in ["promo_count", "promo_stackable_count", "promo_percentage_count", "promo_fixed_count", "promo_online_count", "promo_all_channel_count", *bucket_cols]}
    agg.update({"promo_discount_mean": "mean", "promo_discount_max": "max"})
    daily = long.groupby("Date", as_index=False).agg(agg)
    out = dates[["Date"]].merge(daily, on="Date", how="left")
    promo_starts = promos["start_date"].dropna().sort_values().to_numpy(dtype="datetime64[D]").astype(int)
    promo_ends = promos["end_date"].dropna().sort_values().to_numpy(dtype="datetime64[D]").astype(int)
    d_int = out["Date"].values.astype("datetime64[D]").astype(int)
    next_start = np.where(promo_starts[None, :] >= d_int[:, None], promo_starts[None, :] - d_int[:, None], 9999).min(axis=1)
    last_start = np.where(promo_starts[None, :] <= d_int[:, None], d_int[:, None] - promo_starts[None, :], 9999).min(axis=1)
    next_end = np.where(promo_ends[None, :] >= d_int[:, None], promo_ends[None, :] - d_int[:, None], 9999).min(axis=1)
    out["days_to_next_promo_start"] = pd.Series(next_start).clip(0, 120).to_numpy()
    out["days_since_last_promo_start"] = pd.Series(last_start).clip(0, 120).to_numpy()
    out["days_to_promo_end"] = pd.Series(next_end).clip(0, 120).to_numpy()
    out = out.fillna(0)
    return out

promo_daily = build_promo_features(features[["Date"]], promotions)
features = features.merge(promo_daily, on="Date", how="left")
features[[c for c in features.columns if c.startswith("promo_")]].fillna(0, inplace=True)
features["payday_x_promo"] = ((features["is_private_payday"] | features["is_government_payday"]) * (features.get("promo_count", 0) > 0)).astype(int)
features["tet_x_promo"] = ((features["tet_proximity"] > 0.5) * (features.get("promo_count", 0) > 0)).astype(int)
features["double_day_x_promo"] = (features["is_double_day_window_3"] * (features.get("promo_count", 0) > 0)).astype(int)
features["black_friday_x_promo"] = ((features["black_friday_proximity"] > 0.5) * (features.get("promo_count", 0) > 0)).astype(int)
features.filter(like="promo").head()


## Historical Auxiliary Seasonal Profiles

In [ ]:
def safe_div(a, b):
    return np.where(np.asarray(b, dtype=float) == 0, 0, np.asarray(a, dtype=float) / np.asarray(b, dtype=float))

orders["Date"] = orders["order_date"].dt.normalize()
order_daily = orders.groupby("Date").agg(order_count=("order_id", "count")).reset_index()
status = pd.crosstab(orders["Date"], orders["order_status"], normalize="index").add_prefix("status_share_").reset_index()
device = pd.crosstab(orders["Date"], orders["device_type"], normalize="index").add_prefix("device_share_").reset_index()
order_daily = order_daily.merge(status, on="Date", how="left").merge(device, on="Date", how="left")

items = order_items.merge(orders[["order_id", "Date"]], on="order_id", how="left").merge(products[["product_id", "category", "cogs"]], on="product_id", how="left")
items["gross_item_revenue"] = items["quantity"] * items["unit_price"]
items["discount_rate"] = items["discount_amount"] / (items["gross_item_revenue"] + items["discount_amount"]).replace(0, np.nan)
item_daily = items.groupby("Date").agg(
    item_lines=("order_id", "count"),
    quantity_sum=("quantity", "sum"),
    gross_item_revenue=("gross_item_revenue", "sum"),
    discount_rate=("discount_rate", "mean"),
).reset_index()
cat_rev = items.pivot_table(index="Date", columns="category", values="gross_item_revenue", aggfunc="sum", fill_value=0)
cat_share = cat_rev.div(cat_rev.sum(axis=1).replace(0, np.nan), axis=0).fillna(0).add_prefix("category_share_").reset_index()
item_daily = item_daily.merge(cat_share, on="Date", how="left")

payments_daily = payments.merge(orders[["order_id", "Date"]], on="order_id", how="left").groupby("Date").agg(
    payment_value_sum=("payment_value", "sum"),
    payment_value_mean=("payment_value", "mean"),
    installments_mean=("installments", "mean"),
).reset_index()
return_daily = returns.assign(Date=returns["return_date"].dt.normalize()).groupby("Date").agg(
    return_records=("return_id", "count"),
    returned_units=("return_quantity", "sum"),
    refund_amount=("refund_amount", "sum"),
).reset_index()
web_daily = web.rename(columns={"date": "Date"}).groupby("Date").agg(
    sessions=("sessions", "sum"),
    unique_visitors=("unique_visitors", "sum"),
    page_views=("page_views", "sum"),
    bounce_rate=("bounce_rate", "mean"),
    avg_session_duration_sec=("avg_session_duration_sec", "mean"),
).reset_index()
inventory_daily = inventory.assign(Date=inventory["snapshot_date"].dt.to_period("M").dt.to_timestamp()).groupby("Date").agg(
    stock_on_hand=("stock_on_hand", "sum"),
    units_received=("units_received", "sum"),
    units_sold_inventory=("units_sold", "sum"),
    stockout_days=("stockout_days", "sum"),
    days_of_supply=("days_of_supply", "mean"),
    fill_rate=("fill_rate", "mean"),
    stockout_flag=("stockout_flag", "mean"),
    overstock_flag=("overstock_flag", "mean"),
    sell_through_rate=("sell_through_rate", "mean"),
).reset_index()

aux = features[["Date", "month", "day_of_week", "month_day"]].copy()
for part in [order_daily, item_daily, payments_daily, return_daily, web_daily, inventory_daily]:
    aux = aux.merge(part, on="Date", how="left")
aux_cols = [c for c in aux.columns if c not in ["Date", "month", "day_of_week", "month_day"]]

# Convert same-day operational signals to historical seasonal profiles for every row.
profile_source = aux.merge(features[["Date", "year"]], on="Date", how="left")
profile_source = profile_source[profile_source["Date"].between("2013-01-01", "2022-12-31")]
profile_train = profile_source[profile_source["year"].between(2013, 2018)]
if profile_train.empty:
    profile_train = profile_source
for col in aux_cols:
    md = profile_train.groupby("month_day")[col].median()
    mw = profile_train.groupby(["month", "day_of_week"])[col].median()
    mo = profile_train.groupby("month")[col].median()
    global_val = profile_train[col].median()
    key = list(zip(features["month"], features["day_of_week"]))
    mapped_mw = pd.Series(key, index=features.index).map(mw)
    features[f"profile_{col}"] = (
        features["month_day"].map(md)
        .fillna(mapped_mw)
        .fillna(features["month"].map(mo))
        .fillna(global_val)
        .fillna(0)
    )
print("profile feature count", len([c for c in features.columns if c.startswith("profile_")]))


## Shifted Target History and Artifacts

In [ ]:
for target in ["Revenue", "COGS", "COGS_ratio"]:
    for lag in [1, 2, 3, 7, 14, 28, 56, 91, 182, 365]:
        features[f"lag_{target}_{lag}"] = features[target].shift(lag)
    shifted = features[target].shift(1)
    for window in [7, 14, 28, 56, 91, 365]:
        features[f"roll_{target}_{window}_mean"] = shifted.rolling(window, min_periods=max(2, min(7, window))).mean()
        features[f"roll_{target}_{window}_std"] = shifted.rolling(window, min_periods=max(2, min(7, window))).std()

train_mask = features["Date"].between("2013-01-01", "2022-12-31")
profile_source = features.loc[train_mask].copy()
for col in [c for c in features.columns if c.startswith("lag_") or c.startswith("roll_")]:
    md = profile_source.groupby("month_day")[col].median()
    mo = profile_source.groupby("month")[col].median()
    global_val = profile_source[col].median()
    features[col] = features[col].fillna(features["month_day"].map(md)).fillna(features["month"].map(mo)).fillna(global_val).fillna(0)

train_features = features[features["Date"].between("2013-01-01", "2022-12-31")].copy()
test_features = features[features["Date"].isin(sample["Date"])].copy()
exclude = {"Date", "Revenue", "COGS", "COGS_ratio", "month_day", "days_in_month"}
feature_cols = [c for c in features.columns if c not in exclude and pd.api.types.is_numeric_dtype(features[c])]
for col in feature_cols:
    med = train_features[col].median()
    train_features[col] = train_features[col].fillna(med).fillna(0)
    test_features[col] = test_features[col].fillna(med).fillna(0)
test_features = test_features.drop(columns=[c for c in ["Revenue", "COGS", "COGS_ratio"] if c in test_features.columns])

assert "COGS_ratio" not in feature_cols
assert len(test_features) == 548
assert test_features["Date"].tolist() == sample["Date"].tolist()
train_features.to_parquet(ARTIFACT_DIR / "train_features.parquet", index=False)
test_features.to_parquet(ARTIFACT_DIR / "test_features.parquet", index=False)
(ARTIFACT_DIR / "feature_cols.json").write_text(json.dumps(feature_cols, indent=2), encoding="utf-8")
print(train_features.shape, test_features.shape, len(feature_cols))


## Raw EDA and Leakage Audit

In [ ]:
sales_eda = sales.copy()
sales_eda["year"] = sales_eda["Date"].dt.year
sales_eda["month"] = sales_eda["Date"].dt.month
sales_eda["cogs_ratio"] = sales_eda["COGS"] / sales_eda["Revenue"].replace(0, np.nan)
yearly = sales_eda.groupby("year").agg(days=("Date", "count"), revenue_mean=("Revenue", "mean"), cogs_mean=("COGS", "mean"), ratio_median=("cogs_ratio", "median")).reset_index()
monthly = sales_eda.groupby("month").agg(revenue_mean=("Revenue", "mean"), cogs_mean=("COGS", "mean"), ratio_median=("cogs_ratio", "median")).reset_index()
regime = sales_eda.assign(regime=np.select(
    [sales_eda["year"] <= 2018, sales_eda["year"].between(2019, 2020), sales_eda["year"].eq(2021), sales_eda["year"].eq(2022)],
    ["pre_covid", "transition_shock", "covid_2021", "recovery_2022"], default="other"
)).groupby("regime").agg(days=("Date", "count"), revenue_mean=("Revenue", "mean"), cogs_mean=("COGS", "mean"), ratio_median=("cogs_ratio", "median")).reset_index()
yearly.to_csv(REPORT_DIR / "01_raw_yearly_summary.csv", index=False)
monthly.to_csv(REPORT_DIR / "01_raw_monthly_summary.csv", index=False)
regime.to_csv(REPORT_DIR / "01_raw_regime_summary.csv", index=False)
audit = {
    "sample_submission_loaded_columns": ["Date"],
    "test_target_columns_in_artifact": [c for c in ["Revenue", "COGS", "COGS_ratio"] if c in test_features.columns],
    "same_day_cogs_ratio_in_feature_cols": "COGS_ratio" in feature_cols,
    "target_history_policy": "Revenue/COGS/COGS_ratio lag and rolling features are shifted before prediction date; future missing values are filled from historical profiles only.",
    "same_day_aux_policy": "Operational and transaction CSVs are converted to historical seasonal profile features before modeling.",
}
assert audit["test_target_columns_in_artifact"] == []
assert not audit["same_day_cogs_ratio_in_feature_cols"]
(REPORT_DIR / "01_feature_leakage_audit.json").write_text(json.dumps(audit, indent=2), encoding="utf-8")
display(yearly.tail(6))
display(regime)
audit
